In [ ]:
import gymnasium as gym
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import math

from IPython.display import clear_output
from time import sleep
from tqdm import tqdm

## Custom Envrionments

Racetrack

In [ ]:
class Racetrack:
    def __init__(self):
        self.track = np.empty((40,15), dtype=np.str_)
        self.track[:] = "|"
        self.track[:30, 10:] = '-'
        self.track[30:, -1] = 'F'
        # set random starting position
        self.carv = 0 
        self.carh = np.random.choice(range(9))
        self.velo_vert = 0
        self.velo_hori = 0
        self.track[self.carv, self.carh] = 'C'
        self.crashed = False
        self.at_end = False
    
    def print_state(self):
        for i in reversed(range(self.track.shape[0])):
            print(str(self.track[i, :]))
    
    def step(self, vert_action, hori_action):
        old_v = self.carv
        old_h = self.carh
        self.carv += self.velo_vert
        self.carh += self.velo_hori

        self.velo_vert += vert_action
        if self.velo_vert < 0:
            self.velo_vert = 1
        elif self.velo_vert >5:
            self.velo_vert = 5

        self.velo_hori += hori_action
        if self.velo_hori >5:
            self.velo_hori = 5

        self.spot_check(self.carv, self.carh)
        # reward = distance between previous and current state
        reward = round(math.sqrt((self.carv-old_v)**2+(self.carh-old_h)**2))
         
        if self.at_end:
            reward = 50
        elif self.crashed:
            reward = -10
        else:
            self.track[self.carv, self.carh] = 'C'
        return reward 

    def spot_check(self, v, h):
        # checks if the entering spot is a crash
        if (v >= 40 or v < 0):
            self.crashed = True
            self.at_end = False
        elif (h >= 15 or h < 0):
            self.crashed = True
            self.at_end = False
        elif self.track[v,h] == '-':
            self.crashed = True
            self.at_end = False
        elif self.track[v,h] == 'F':
            self.crashed = False
            self.at_end = True
    
    def reset(self):
        # resets to start 
        self.track[:] = "|"
        self.track[:30, 10:] = '-'
        self.track[30:, -1] = 'F'
        # set random starting position
        self.carv = 0 
        self.carh = np.random.choice(range(9))
        self.velo_vert = 0
        self.velo_hori = 0
        self.track[self.carv, self.carh] = 'C'
        self.crashed = False
        self.at_end = False

def get_action():
    vert = np.random.choice([-1,0,1])
    hori = np.random.choice([-1, 0,1])
    return (vert, hori)

In [ ]:
data = {
    't':[],
    'cum_reward':[],
    'finish':[],
    'crash':[],
    'final_x':[],
    'final_y':[],
}
cum_heat = np.zeros((40,15))
fin_heat = np.zeros((40,15))
crash_heat = np.zeros((40,15))
track = Racetrack()
for i in tqdm(range(50000)):
    cum_reward = 0
    t = 0 
    while (not track.crashed or track.at_end):
        t+=1
        v, h = get_action()
        cum_reward += track.step(v,h)
        # print("t: ",t ,"Cum Reward: ", cum_reward, "\nCrashed: ", track.crashed, "\nAt End: ", track.at_end)       
        clear_output(wait=True)
    cum_heat += np.where(track.track == 'C', 1, 0)
    if track.at_end:
        fin_heat += np.where(track.track == 'C', 1, 0)
        data['finish'].append(1)
        data['crash'].append(0)
    elif track.crashed:
        crash_heat += np.where(track.track == 'C', 1, 0)
        data['finish'].append(0)
        data['crash'].append(1)
    data['t'].append(t)
    data['cum_reward'].append(cum_reward)
    data['final_x'].append(track.carh)
    data['final_y'].append(track.carv)
    track.reset()

In [ ]:
# Plots
ax = sns.heatmap(data=cum_heat)
ax.invert_yaxis()
ax.set(title='Heat map of Crashes')
fig = ax.get_figure()
fig.savefig('crashes_heatmap.png', format='png')

ax = sns.lineplot(
    data=df,
    x='t', 
    y='cum_reward')
ax.set(title="Cumlative Rewards vs Number steps")
fig = ax.get_figure()
fig.savefig('images/reward_v_steps.png', format='png')

In [ ]:
## Chase
class ChaseEnvrionment:
    def __init__(self, n):
        self.n = n
        self.player_x = 0
        self.player_y = 0
        self.player_h = 0
        self.player_v = 0
        chocies = np.random.choice(range(1, self.n), 2)
        self.predator_x = chocies[0]
        self.predator_y = chocies[1]
        self.base = [self.n, self.n]
        self.caught = False
        self.at_end = False

    def reset(self):
        self.player_x = 0
        self.player_y = 0
        self.player_h = 0
        self.player_v = 0
        chocies = np.random.choice(range(1, self.n), 2)
        self.predator_x = chocies[0]
        self.predator_y = chocies[1]
        self.base = [self.n, self.n]
        self.caught = False
        self.at_end = False

    def player_speed(self, action):
        value = 0
        match action:
            case 1:
                # Vh + U(|0.5, 1.5)
                value = random.uniform(0.5, 1.5)
            case -1:
                value = random.uniform(-0.5, -1.5)
        return value

    def check_spot(self):
        # check x location
        if self.player_x < 0:
            self.player_x = 0
        elif self.player_x > self.n:
            self.player_x = self.n
        # check y locagtion
        if self.player_y < 0:
            self.player_y = 0
        elif self.player_y > self.n:
            self.player_y = self.n
        # check predator
        if (round(self.player_x) == round(self.predator_x)) and (round(self.player_y) == round(self.predator_y)):
            self.caught = True
        # check base
        if self.base_dist()<= 0.5:
            self.at_end = True

    def sign(self, val):
        return -1 if val < 0 else 1
    def get_pred_move(self):
        # moves a distance no more than 4 toward the player
        x_val = self.player_x - self.predator_x
        y_val = self.player_y - self.predator_y
        
        self.predator_x = x_val if x_val > 0 else 1
        self.predator_y = y_val if y_val > 0 else 1
        if self.predator_x > self.n:
            self.predator_x = n
        if self.predator_y > self.n:
            self.predator_y = n


    def pred_dist(self):
        return np.sqrt((self.player_x-self.predator_x)**2 +(self.player_y-self.predator_y)**2)
    def base_dist(self):
        return np.sqrt((self.n-self.player_x)**2 +(self.n-self.player_y)**2)

    def step(self, action):
        player_first = np.random.choice([True, False])
        if player_first:
            # do player stuff
            self.player_x+=self.player_h
            self.player_y+=self.player_v
            self.get_pred_move()
        else:
            self.get_pred_move()
            self.player_x+=self.player_h
            self.player_y+=self.player_v
        self.check_spot()

        self.player_h += self.player_speed(action)
        self.player_v += self.player_speed(action)
        if self.player_h > 5:
            self.player_h = 5
        if self.player_v > 5:
            self.player_v =5 

        reward = self.pred_dist()/(self.base_dist()+0.1) # prevents divide by zero
        return reward
        
    def show(self):
        print(f"Player Location: {self.player_x},{self.player_y}")
        print(f"Predator Location: {self.predator_x},{self.predator_y}")
        print(f"Distance to Predator: {self.pred_dist()}")
        print(f"Distance to Base: {self.base_dist()}")


In [ ]:
data = {
    't':[],
    'cum_reward':[],
    'escape':[],
    'caught':[],
    'pred_pos':[],
    'player_pos':[]
}
n = 25
for i in tqdm(range(1000)):
    t=0
    cum_reward = 0 
    chase = ChaseEnvrionment(n)
    while(not chase.at_end and not chase.caught):
        action = np.random.choice([-1,0,1])
        cum_reward += chase.step(action=action)
        t+=1
    if chase.at_end:
        data['escape'].append(1)
        data['caught'].append(0)
    else:
        data['escape'].append(0)
        data['caught'].append(1)
    data['t'].append(t)
    data['cum_reward'].append(cum_reward)
    data['pred_pos'].append((chase.predator_x, chase.predator_y))
    data['player_pos'].append((chase.player_x, chase.player_y))

df = pd.DataFrame(data=data)
df.head()

In [ ]:
# plots
g = sns.scatterplot(
    data=df,
    x='player_x',
    y='player_y',
    hue='caught'
)
g.set(title="Player Location at End")
fig = g.figure
fig.savefig("images/chase_locations.png", format='png')

g1 = sns.histplot(
    data=df, 
    x='t',hue="escape",
    multiple="stack",
    edgecolor=".3",
    linewidth=.5,
    log_scale=True,
)
g1.set(title="Number of Elapsed Steps")
fig = g1.figure
fig.savefig("images/chase_steps.png", format='png')

## Custom Harder Chase with Hole

There is a hole, if you fall in it, you get sent back to the start

In [ ]:
## Chase
class ChaseEnvrionmentHole:
    def __init__(self, n):
        self.n = n
        self.player_x = 0
        self.player_y = 0
        self.player_h = 0
        self.player_v = 0
        chocies = np.random.choice(range(1, self.n), 2)
        self.predator_x = chocies[0]
        self.predator_y = chocies[1]
        self.base = [self.n, self.n]
        self.caught = False
        self.at_end = False
        
        self.hole_x1 = 5
        self.hole_x2 = 8
        self.hole_y1 = 11
        self.hole_y2 = 14

    def reset(self):
        self.player_x = 0
        self.player_y = 0
        self.player_h = 0
        self.player_v = 0
        chocies = np.random.choice(range(1, self.n), 2)
        self.predator_x = chocies[0]
        self.predator_y = chocies[1]
        self.base = [self.n, self.n]
        self.caught = False
        self.at_end = False

    def player_speed(self, action):
        value = 0
        match action:
            case 1:
                # Vh + U(|0.5, 1.5)
                value = random.uniform(0.5, 1.5)
            case -1:
                value = random.uniform(-0.5, -1.5)
        return value

    def check_spot(self):
        # if touching icey river, slide away from the base side
        if (self.hole_x1 < self.player_x <= self.hole_x2) and (self.hole_y1 < self.player_y <= self.hole_y2):
            self.player_y = 0
            self.player_x = 0
        # check x location
        if self.player_x < 0:
            self.player_x = 0
        elif self.player_x > self.n:
            self.player_x = self.n
        # check y locagtion
        if self.player_y < 0:
            self.player_y = 0
        elif self.player_y > self.n:
            self.player_y = self.n
        # check predator
        if (round(self.player_x) == round(self.predator_x)) and (round(self.player_y) == round(self.predator_y)):
            self.caught = True
        # check base
        if self.base_dist()<= 0.5:
            self.at_end = True

    def sign(self, val):
        return -1 if val < 0 else 1
    def get_pred_move(self):
        # moves a distance no more than 4 toward the player
        x_val = self.player_x - self.predator_x
        y_val = self.player_y - self.predator_y
        
        self.predator_x = x_val if x_val > 0 else 1
        self.predator_y = y_val if y_val > 0 else 1
        if self.predator_x > self.n:
            self.predator_x = n
        if self.predator_y > self.n:
            self.predator_y = n


    def pred_dist(self):
        return np.sqrt((self.player_x-self.predator_x)**2 +(self.player_y-self.predator_y)**2)
    def base_dist(self):
        return np.sqrt((self.n-self.player_x)**2 +(self.n-self.player_y)**2)

    def step(self, action):
        player_first = np.random.choice([True, False])
        if player_first:
            # do player stuff
            self.player_x+=self.player_h
            self.player_y+=self.player_v
            self.get_pred_move()
        else:
            self.get_pred_move()
            self.player_x+=self.player_h
            self.player_y+=self.player_v
        self.check_spot()

        self.player_h += self.player_speed(action)
        self.player_v += self.player_speed(action)
        if self.player_h > 5:
            self.player_h = 5
        if self.player_v > 5:
            self.player_v =5 

        reward = self.pred_dist()/(self.base_dist()+0.1) # prevents divide by zero
        return reward
        
    def show(self):
        print(f"Player Location: {self.player_x},{self.player_y}")
        print(f"Predator Location: {self.predator_x},{self.predator_y}")
        print(f"Distance to Predator: {self.pred_dist()}")
        print(f"Distance to Base: {self.base_dist()}")
        print(f"The Hole is at X:{self.hole_x1}-{self.hole_x2} and Y: {self.hole_y1}-{self.hole_x2}")


data = {
    't':[],
    'cum_reward':[],
    'escape':[],
    'caught':[],
    'pred_pos':[],
    'player_pos':[]
}
n = 25
for i in tqdm(range(1000)):
    t=0
    cum_reward = 0 
    chase = ChaseEnvrionment(n)
    while(not chase.at_end and not chase.caught):
        action = np.random.choice([-1,0,1])
        cum_reward += chase.step(action=action)
        t+=1
    if chase.at_end:
        data['escape'].append(1)
        data['caught'].append(0)
    else:
        data['escape'].append(0)
        data['caught'].append(1)
    data['t'].append(t)
    data['cum_reward'].append(cum_reward)
    data['pred_pos'].append((chase.predator_x, chase.predator_y))
    data['player_pos'].append((chase.player_x, chase.player_y))

dt = pd.DataFrame(data=data)
dt.head()

In [ ]:
g = sns.scatterplot(
    data=dt,
    x='player_x',
    y='player_y',
    hue='caught'
)
g.set(title="Player Location at End with added Hole")
fig = g.figure
fig.savefig("images/chase_hard_locations.png", format='png')

g1 = sns.histplot(
    data=dt, 
    x='t',hue="escape",
    multiple="stack",
    edgecolor=".3",
    linewidth=.5,
    log_scale=True,
)
g1.set(title="Number of Elapsed Steps with Hole")
fig = g1.figure
fig.savefig("images/chase_hard_steps.png", format='png')

# Gynasium Envrionment

In [ ]:
data = {
    'episode':[],
    'action':[],
    'observation':[],
    'reward':[],
    'terminated':[],
    'truncated':[],
    'info':[]
}

env = gym.make("LunarLander-v3", render_mode="human")
observation, info = env.reset(seed=67)
for i in range(1000):
    action = env.action_space.sample()
    observation, reward, terminated, truncated, info = env.step(action)
    data['episode'].append(i)
    data['action'].append(action)
    data['observation'].append(observation)
    data['reward'].append(reward)
    data['terminated'].append(terminated)
    data['truncated'].append(truncated)
    data['info'].append(info)
    if terminated or truncated:
        observation, info = env.reset()
env.close()

In [ ]:
df = pd.DataFrame(data=data)
df.head()

In [ ]:
df['action'].unique(), df['observation'][0], df['reward'].mean(), df['terminated'].sum(), df['truncated'].sum()